In [0]:
-- Pergunta 1: associação entre atraso e satisfação.
WITH pedidos AS (
  SELECT
    order_id,
    delivery_status,
    MAX(CAST(review_score AS INT)) AS review_score
  FROM ${catalog}.olist_gold.fato_vendas
  GROUP BY order_id, delivery_status
)
SELECT
  delivery_status,
  COUNT(*) AS total_pedidos,
  ROUND(AVG(review_score), 2) AS nota_media,
  ROUND(
    100.0 * SUM(CASE WHEN review_score IN (1, 2) THEN 1 ELSE 0 END) /
    NULLIF(SUM(CASE WHEN review_score IS NOT NULL THEN 1 ELSE 0 END), 0),
    2
  ) AS percentual_notas_1_2
FROM pedidos
GROUP BY delivery_status
ORDER BY total_pedidos DESC;

delivery_status,total_pedidos,nota_media,percentual_notas_1_2
adiantado_ou_no_prazo,89941,4.29,9.27
atraso_1_7_dias,3672,2.72,49.33
atraso_mais_de_7_dias,2863,1.7,79.30
nao_entregue,2190,1.76,77.38


In [0]:
-- Apoio para a conclusao: compara apenas pedidos avaliados.
WITH pedidos AS (
  SELECT 
    order_id, 
    delivery_status,
    MAX(CAST(review_score AS INT)) AS review_score
  FROM ${catalog}.olist_gold.fato_vendas
  GROUP BY order_id, delivery_status
)
SELECT
  ROUND(AVG(CASE WHEN delivery_status = 'adiantado_ou_no_prazo' THEN review_score END), 2)
    AS nota_no_prazo,
  ROUND(AVG(CASE WHEN delivery_status IN ('atraso_1_7_dias', 'atraso_mais_de_7_dias')
                THEN review_score END), 2) AS nota_com_atraso,
  ROUND(
    AVG(CASE WHEN delivery_status IN ('atraso_1_7_dias', 'atraso_mais_de_7_dias')
             THEN review_score END)
    - AVG(CASE WHEN delivery_status = 'adiantado_ou_no_prazo' THEN review_score END),
    2
  ) AS diferenca_pontos
FROM pedidos
WHERE review_score IS NOT NULL;

nota_no_prazo,nota_com_atraso,diferenca_pontos
4.29,2.27,-2.02


### Conclusao da Pergunta 1
Compare `nota_com_atraso` com `nota_no_prazo` na consulta anterior. Se `diferenca_pontos` for negativa, os pedidos atrasados apresentam menor satisfacao; quanto mais distante de zero, maior a associacao observada. Registre tambem o `percentual_notas_1_2` dos grupos com atraso. Esta analise indica associacao, nao prova causalidade.

In [0]:
-- Pergunta 2: estados líderes em receita e sua forma de pagamento predominante.
WITH state_sales AS (
  SELECT
    c.customer_state,
    COUNT(DISTINCT f.order_id) AS total_pedidos,
    ROUND(SUM(f.price + f.freight_value), 2) AS receita_brl
  FROM ${catalog}.olist_gold.fato_vendas f
  JOIN ${catalog}.olist_gold.dim_clientes c USING (customer_id)
  GROUP BY c.customer_state
),
state_payment AS (
  SELECT
    customer_state,
    payment_type,
    COUNT(DISTINCT order_id) AS pedidos_com_pagamento,
    ROW_NUMBER() OVER (
      PARTITION BY customer_state
      ORDER BY COUNT(DISTINCT order_id) DESC, payment_type
    ) AS row_number
  FROM (
    SELECT DISTINCT c.customer_state, f.order_id, f.payment_type
    FROM ${catalog}.olist_gold.fato_vendas f
    JOIN ${catalog}.olist_gold.dim_clientes c USING (customer_id)
    WHERE f.payment_type IS NOT NULL
  ) orders_with_payment
  GROUP BY customer_state, payment_type
)
SELECT
  s.customer_state,
  s.total_pedidos,
  s.receita_brl,
  p.payment_type AS pagamento_predominante,
  p.pedidos_com_pagamento
FROM state_sales s
LEFT JOIN state_payment p
  ON p.customer_state = s.customer_state AND p.row_number = 1
ORDER BY s.receita_brl DESC;

customer_state,total_pedidos,receita_brl,pagamento_predominante,pedidos_com_pagamento
SP,41375,5921678.12,credit_card,31833
RJ,12762,2129681.98,credit_card,10188
MG,11544,1856161.49,credit_card,8965
RS,5432,885826.76,credit_card,3948
PR,4998,800935.44,credit_card,3747
BA,3358,611506.67,credit_card,2630
SC,3612,610213.60,credit_card,2690
DF,2125,353229.44,credit_card,1687
GO,2007,347706.93,credit_card,1506
ES,2025,324801.91,credit_card,1560


### Conclusao da Pergunta 2
Na tabela acima, o primeiro estado por `receita_brl` e o estado lider em faturamento no periodo analisado. A coluna `pagamento_predominante` mostra o meio de pagamento mais usado nesse estado. Para a resposta final, registre: estado lider, receita, quantidade de pedidos e forma de pagamento predominante. Compare os tres primeiros estados para identificar concentracao regional.


## Resposta executiva
A Olist deve priorizar a reducao de atrasos quando a diferenca entre as notas atrasadas e no prazo for negativa, monitorando especialmente o percentual de notas 1 e 2. Comercialmente, deve concentrar a leitura de receita nos estados lideres e garantir boa disponibilidade do pagamento predominante, sem deixar de acompanhar os demais meios de pagamento. Substitua os campos acima pelos valores exibidos nas consultas antes de entregar o relatorio.
